In [1]:
import numpy as np
import pandas as pd
import re
import nltk
#import spacy
import string


In [7]:
##Чтение датасета с текстом
df = pd.read_csv("C:/Users/psitn/OneDrive/Desktop/python/zamok.test.csv", encoding='UTF8', sep="\t")
df.head()

,text
0,отвозя их в собственным зАмок с соблазнительны...
1,откладывая замОк в сторону
2,отключающей замОк
3,откроет замОк
4,открой замОк


In [8]:
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('russian'))
# функция, удаляющая стопслова из текстов
def stopwords(text):
    return " ".join([word for word in str(text).split() if word not in STOPWORDS])
# Удаление эмодзи
def emoji(string):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', string)
#Удаление URL
def remove_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)
from bs4 import BeautifulSoup

def html(text):
    return BeautifulSoup(text, "lxml").text
#Можно выполнить при помощи NLTK. Заодно проведем лемматизацию
from pymorphy3 import MorphAnalyzer
from nltk import sent_tokenize, word_tokenize, regexp_tokenize

def tokenize_lemmas(sent, pat=r"(?u)\b\w\w+\b", morph=MorphAnalyzer()):
    return [morph.parse(tok)[0].normal_form 
            for tok in regexp_tokenize(sent, pat)]


In [9]:
#df['text_ready']  = df['text'].str.lower()
df['text_ready'] = df['text'].str.replace('\d+', '') 
df['text_ready'] = df['text_ready'].str.replace('[^\w\s]','')
df["text_ready"] = df["text_ready"].apply(stopwords)
df['text_ready'] = df['text_ready'].apply(emoji)
df['text_ready'] = df['text_ready'].apply(remove_urls)
df['text_ready'] = df['text_ready'].apply(html)
df["text_ready"] = df["text_ready"].map(lambda x: " ".join(tokenize_lemmas(x)))
df[['text_ready']].head()

<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
C:\Users\psitn\AppData\Local\Temp\ipykernel_17624\3602921777.py:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df['text_ready'] = df['text'].str.replace('\d+', '')
C:\Users\psitn\AppData\Local\Temp\ipykernel_17624\3602921777.py:3: SyntaxWarning: "\w" is an invalid escape sequence. S

,text_ready
0,отвозить собственный замок соблазнительный сук...
1,откладывать замок сторона
2,отключать замок
3,открыть замок
4,открыть замок


In [10]:
import sys
print(sys.executable)
print(sys.version)

C:\Users\psitn\anaconda3\python.exe
3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]


In [11]:
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

In [12]:
w2v_model = Word2Vec(
    min_count=10,
    window=4,
    vector_size=300,
    negative=10,
    alpha=0.03,
    min_alpha=0.0007,
    sample=6e-5,
    sg=1)
#vector_size — размер векторного представления слова (word embedding).
#negative — сколько неконтекстных слов учитывать в обучении, используя negative sampling.
#alpha — начальный learning_rate, используемый в алгоритме обратного распространения ошибки (Backpropogation).
#min_alpha — минимальное значение learning_rate, на которое может опуститься в процессе обучения.
#sg — если 1, то используется реализация Skip-gram; если 0, то CBOW.

In [13]:
#Получаем лист слов
from nltk.tokenize import sent_tokenize
from gensim.utils import simple_preprocess

class MySentences(object):
    def __init__(self, docs):
        self.corpus = docs
    def __iter__(self):
        for doc in self.corpus:
            doc_sentences = sent_tokenize(doc)
            for sent in doc_sentences:
                yield simple_preprocess(sent) # yields a tokenized 

In [14]:
sentences = MySentences(df['text_ready'].tolist())

In [15]:
#Получаем словарь
w2v_model.build_vocab(sentences)

In [16]:
#Обучение
w2v_model.train(sentences, total_examples=w2v_model.corpus_count, epochs=6, report_delay=1)

(231, 20946)

In [17]:
#для сохранения оперативной памяти можно написать следующее
w2v_model.init_sims(replace=True)

C:\Users\psitn\AppData\Local\Temp\ipykernel_17624\232342886.py:2: DeprecationWarning: Call to deprecated `init_sims` (Gensim 4.0.0 implemented internal optimizations that make calls to init_sims() unnecessary. init_sims() is now obsoleted and will be completely removed in future versions. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  w2v_model.init_sims(replace=True)


In [19]:
w2v_model.wv.most_similar(positive=["замок"], topn=35)

[('вернуться', 0.1964678168296814),
 ('дверной', 0.0871828943490982),
 ('пытаться', 0.08629453927278519),
 ('дверь', 0.08554419875144958),
 ('сложить', 0.0684761255979538),
 ('ключ', 0.04970923066139221),
 ('щёлкнуть', 0.0347759947180748),
 ('сцепить', 0.034541837871074677),
 ('кодовый', 0.025762731209397316),
 ('палец', 0.018436718732118607),
 ('рука', 0.015086483210325241),
 ('минута', 0.012463890016078949),
 ('первый', 0.00244798231869936),
 ('посадить', 0.001919567584991455),
 ('свой', -0.006010323762893677),
 ('который', -0.00740312784910202),
 ('электронный', -0.011404000222682953),
 ('открыть', -0.013681311160326004),
 ('зажигание', -0.0141326067969203),
 ('человек', -0.014559464529156685),
 ('открывать', -0.01951959729194641),
 ('закрыть', -0.03953927382826805),
 ('отпереть', -0.05576383322477341),
 ('запереть', -0.06862281262874603)]

In [21]:
w2v_model.wv.most_similar(positive=["дверь", "замок"], topn=35)

[('вернуться', 0.24332091212272644),
 ('дверной', 0.10560506582260132),
 ('щёлкнуть', 0.057803861796855927),
 ('кодовый', 0.04475771263241768),
 ('электронный', 0.04007924720644951),
 ('сцепить', 0.037231460213661194),
 ('ключ', 0.016259722411632538),
 ('первый', 0.013987661339342594),
 ('открывать', 0.006131913512945175),
 ('пытаться', 0.00451447069644928),
 ('человек', 0.0011253319680690765),
 ('свой', -0.00045913271605968475),
 ('сложить', -0.0053301043808460236),
 ('рука', -0.01329705212265253),
 ('закрыть', -0.016286328434944153),
 ('палец', -0.016718540340662003),
 ('зажигание', -0.028249528259038925),
 ('который', -0.03664730489253998),
 ('посадить', -0.04226285591721535),
 ('отпереть', -0.047720324248075485),
 ('минута', -0.05458413064479828),
 ('открыть', -0.06564059108495712),
 ('запереть', -0.09497654438018799)]

In [22]:
#Векторы можно складывать и вычитать. Например, рассмотрим такой вариант: “товар” + “продавец” — “ужасный”:
w2v_model.wv.most_similar(positive=["закрыть", "отпереть"], negative=["дверь"])

[('который', 0.17694127559661865),
 ('открыть', 0.12902861833572388),
 ('пытаться', 0.07752510905265808),
 ('запереть', 0.07734551280736923),
 ('сцепить', 0.06931214779615402),
 ('свой', 0.059020597487688065),
 ('щёлкнуть', 0.037126325070858),
 ('минута', 0.0331665463745594),
 ('рука', 0.03085978701710701),
 ('посадить', 0.01799154281616211)]

In [23]:
#определить наиболее близкое слово из списка к данному слово
w2v_model.wv.most_similar_to_given("вернуться", ["свой", "дверь", "минута", "рука"])

'дверь'

In [26]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

def tsne_scatterplot(model, word, list_names):
    """Plot in seaborn the results from the t-SNE dimensionality reduction 
    algorithm of the vectors of a query word,
    its list of most similar words, and a list of words."""
    vectors_words = [model.wv.word_vec(word)]
    word_labels = [word]
    color_list = ['red']

    close_words = model.wv.most_similar(word)
    for wrd_score in close_words:
        wrd_vector = model.wv.word_vec(wrd_score[0])
        vectors_words.append(wrd_vector)
        word_labels.append(wrd_score[0])
        color_list.append('blue')

    # adds the vector for each of the words from list_names to the array
    for wrd in list_names:
        wrd_vector = model.wv.word_vec(wrd)
        vectors_words.append(wrd_vector)
        word_labels.append(wrd)
        color_list.append('green')

    # t-SNE reduction
    Y = (TSNE(n_components=2, random_state=0, perplexity=15, init="pca")
        .fit_transform(vectors_words))
    # Sets everything up to plot
    df = pd.DataFrame({"x": [x for x in Y[:, 0]],
                    "y": [y for y in Y[:, 1]],
                    "words": word_labels,
                    "color": color_list})
    fig, _ = plt.subplots()
    fig.set_size_inches(9, 9)
    # Basic plot
    p1 = sns.regplot(data=df,
                    x="x",
                    y="y",
                    fit_reg=False,
                    marker="o",
                    scatter_kws={"s": 40,
                                "facecolors": df["color"]}
    )
    # Adds annotations one by one with a loop
    for line in range(0, df.shape[0]):
        p1.text(df["x"][line],
                df["y"][line],
                " " + df["words"][line].title(),
                horizontalalignment="left",
                verticalalignment="bottom", size="medium",
                color=df["color"][line],
                weight="normal"
        ).set_size(15)

    plt.xlim(Y[:, 0].min()-50, Y[:, 0].max()+50)
    plt.ylim(Y[:, 1].min()-50, Y[:, 1].max()+50)
    plt.title('t-SNE visualization for {}'.format(word.title()))

In [27]:
#Bag of words

from sklearn.feature_extraction.text import CountVectorizer

# create the vocabulary
vectorizer = CountVectorizer()
#matrix = vectorizer.fit_transform(df.text_ready)
# fit the vocabulary to the text data
vectorizer.fit(df['text_ready'])

# create the bag-of-words model
bow_model = vectorizer.transform(df['text_ready'])

# print the bag-of-words model

In [28]:
print(bow_model)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3507 stored elements and shape (878, 1357)>
  Coords	Values
  (0, 274)	1
  (0, 605)	1
  (0, 1047)	1
  (0, 1073)	1
  (0, 1075)	1
  (0, 1142)	1
  (1, 274)	1
  (1, 611)	1
  (1, 1133)	1
  (2, 274)	1
  (2, 612)	1
  (3, 274)	1
  (3, 618)	1
  (4, 274)	1
  (4, 618)	1
  (5, 196)	1
  (5, 274)	1
  (5, 618)	1
  (6, 274)	1
  (6, 618)	1
  (7, 274)	1
  (7, 343)	1
  (7, 427)	1
  (7, 618)	1
  (8, 274)	1
  :	:
  (871, 809)	1
  (871, 1219)	1
  (872, 274)	1
  (872, 627)	1
  (873, 274)	1
  (873, 422)	1
  (873, 1242)	1
  (874, 274)	1
  (874, 957)	1
  (874, 1154)	1
  (874, 1246)	1
  (875, 274)	1
  (875, 537)	1
  (875, 1253)	1
  (876, 93)	1
  (876, 274)	1
  (876, 415)	1
  (876, 417)	1
  (876, 487)	1
  (876, 802)	1
  (876, 1174)	1
  (876, 1209)	1
  (876, 1278)	1
  (877, 274)	1
  (877, 1355)	1


In [29]:
print(vectorizer.get_feature_names_out())

['1010' '1515' 'iv' ... 'юникон' 'язычок' 'ящик']


In [30]:
#tf-idf

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['text_ready'])

In [31]:
# print the vocabulary
print(X_tfidf)



<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3507 stored elements and shape (878, 1357)>
  Coords	Values
  (0, 605)	0.45748471864570645
  (0, 1075)	0.3983243403700883
  (0, 274)	0.06456507330878837
  (0, 1073)	0.45748471864570645
  (0, 1142)	0.45748471864570645
  (0, 1047)	0.45748471864570645
  (1, 274)	0.10215204437340573
  (1, 611)	0.723812378493654
  (1, 1133)	0.6823932887783211
  (2, 274)	0.13974570327103572
  (2, 612)	0.9901874259034416
  (3, 274)	0.26303431813102496
  (3, 618)	0.9647864776650567
  (4, 274)	0.26303431813102496
  (4, 618)	0.9647864776650567
  (5, 274)	0.1839215486578278
  (5, 618)	0.6746078776226361
  (5, 196)	0.7148965487319048
  (6, 274)	0.26303431813102496
  (6, 618)	0.9647864776650567
  (7, 274)	0.10263957428995177
  (7, 618)	0.37647282701307494
  (7, 343)	0.6856500727729082
  (7, 427)	0.614505741239955
  (8, 274)	0.12436057173467684
  :	:
  (871, 185)	0.4228819967149669
  (871, 151)	0.4228819967149669
  (872, 274)	0.19348350619697582
  (872, 6